# SC-7-DAO-Governance - Gouvernance DAO

**Navigation** : [Index](../README.md) | [<< DeFi](SC-6-DeFi-Primitives.ipynb) | [Account Abstraction >>](SC-8-Account-Abstraction.ipynb)

---

## Objectifs d'apprentissage

1. Implementer un systeme de **vote** pondere
2. Creer et executer des **propositions**
3. Utiliser un **timelock** pour la securite

### Duree estimee : 45 minutes

---

## 1. Governance Token

Le token de gouvernance donne des droits de vote.

In [1]:
# Governance Token avec checkpoint de votes
GOVERNANCE_TOKEN = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

contract GovernanceToken {
    string public constant name = "Governance Token";
    string public constant symbol = "GOV";
    uint8 public constant decimals = 18;

    uint256 private _totalSupply;
    mapping(address => uint256) private _balances;
    mapping(address => mapping(address => uint256)) private _allowances;

    // Checkpoints pour voting power
    struct Checkpoint {
        uint32 fromBlock;
        uint224 votes;
    }

    mapping(address => Checkpoint[]) private _checkpoints;
    Checkpoint[] private _totalSupplyCheckpoints;

    event Transfer(address indexed from, address indexed to, uint256 value);
    event Approval(address indexed owner, address indexed spender, uint256 value);
    event DelegateChanged(address indexed delegator, address indexed fromDelegate, address indexed toDelegate);
    event DelegateVotesChanged(address indexed delegate, uint256 previousBalance, uint256 newBalance);

    constructor(uint256 initialSupply) {
        _mint(msg.sender, initialSupply);
    }

    // Delegation: delegate voting power to yourself or another
    function delegate(address delegatee) public {
        _delegate(msg.sender, delegatee);
    }

    function _delegate(address delegator, address delegatee) internal {
        address currentDelegate = delegates(delegator);
        uint256 delegatorBalance = _balances[delegator];

        _moveVotingPower(currentDelegate, delegatee, delegatorBalance);
        emit DelegateChanged(delegator, currentDelegate, delegatee);
    }

    function _moveVotingPower(address src, address dst, uint256 amount) internal {
        if (src != address(0)) {
            _writeCheckpoint(_checkpoints[src], _subtract, amount);
        }
        if (dst != address(0)) {
            _writeCheckpoint(_checkpoints[dst], _add, amount);
        }
    }

    function _writeCheckpoint(
        Checkpoint[] storage ckpts,
        function(uint224, uint224) view returns (uint224) op,
        uint256 delta
    ) internal {
        uint256 pos = ckpts.length;
        uint224 oldVotes = pos > 0 ? ckpts[pos - 1].votes : 0;
        uint224 newVotes = op(oldVotes, uint224(delta));

        if (pos > 0 && ckpts[pos - 1].fromBlock == block.number) {
            ckpts[pos - 1].votes = newVotes;
        } else {
            ckpts.push(Checkpoint({fromBlock: uint32(block.number), votes: newVotes}));
        }
    }

    function _add(uint224 a, uint224 b) internal pure returns (uint224) {
        return a + b;
    }

    function _subtract(uint224 a, uint224 b) internal pure returns (uint224) {
        return a - b;
    }

    // Get current votes
    function getVotes(address account) public view returns (uint256) {
        uint256 pos = _checkpoints[account].length;
        return pos > 0 ? _checkpoints[account][pos - 1].votes : 0;
    }

    // Get votes at a specific block
    function getPastVotes(address account, uint256 blockNumber) 
        public view returns (uint256) 
    {
        require(blockNumber < block.number, "Block not yet mined");
        return _checkpointsLookup(_checkpoints[account], blockNumber);
    }

    function _checkpointsLookup(Checkpoint[] storage ckpts, uint256 blockNumber)
        internal view returns (uint256)
    {
        // Binary search
        uint256 low = 0;
        uint256 high = ckpts.length;

        while (low < high) {
            uint256 mid = (low + high) / 2;
            if (ckpts[mid].fromBlock > blockNumber) {
                high = mid;
            } else {
                low = mid + 1;
            }
        }
        return high == 0 ? 0 : ckpts[high - 1].votes;
    }

    // Simplified delegate tracking
    mapping(address => address) private _delegates;

    function delegates(address account) public view returns (address) {
        return _delegates[account] == address(0) ? account : _delegates[account];
    }

    // ERC20 functions
    function totalSupply() public view returns (uint256) {
        return _totalSupply;
    }

    function balanceOf(address account) public view returns (uint256) {
        return _balances[account];
    }

    function transfer(address to, uint256 amount) public returns (bool) {
        _transfer(msg.sender, to, amount);
        return true;
    }

    function _transfer(address from, address to, uint256 amount) internal {
        require(from != address(0), "Transfer from zero");
        require(to != address(0), "Transfer to zero");
        require(_balances[from] >= amount, "Insufficient balance");

        _balances[from] -= amount;
        _balances[to] += amount;

        // Move voting power
        _moveVotingPower(delegates(from), delegates(to), amount);

        emit Transfer(from, to, amount);
    }

    function _mint(address to, uint256 amount) internal {
        _totalSupply += amount;
        _balances[to] += amount;
        _moveVotingPower(address(0), delegates(to), amount);
        emit Transfer(address(0), to, amount);
    }
}
'''

print("Governance Token avec delegation:")
print(GOVERNANCE_TOKEN)

Governance Token avec delegation:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

contract GovernanceToken {
    string public constant name = "Governance Token";
    string public constant symbol = "GOV";
    uint8 public constant decimals = 18;

    uint256 private _totalSupply;
    mapping(address => uint256) private _balances;
    mapping(address => mapping(address => uint256)) private _allowances;

    // Checkpoints pour voting power
    struct Checkpoint {
        uint32 fromBlock;
        uint224 votes;
    }

    mapping(address => Checkpoint[]) private _checkpoints;
    Checkpoint[] private _totalSupplyCheckpoints;

    event Transfer(address indexed from, address indexed to, uint256 value);
    event Approval(address indexed owner, address indexed spender, uint256 value);
    event DelegateChanged(address indexed delegator, address indexed fromDelegate, address indexed toDelegate);
    event DelegateVotesChanged(address indexed delegate, uint256 previousBalance, u

---

## 2. Governor Contract

Le governor gere le cycle de vie des propositions.

In [2]:
# Governor simplifie
GOVERNOR_CONTRACT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

interface IGovernanceToken {
    function getVotes(address account) external view returns (uint256);
    function getPastVotes(address account, uint256 blockNumber) external view returns (uint256);
}

contract SimpleGovernor {
    IGovernanceToken public immutable token;
    
    uint256 public constant VOTING_DELAY = 1;        // Blocs avant vote
    uint256 public constant VOTING_PERIOD = 50400;   // ~1 semaine (12s blocks)
    uint256 public constant PROPOSAL_THRESHOLD = 100 * 1e18;  // Min tokens pour proposer
    uint256 public constant QUORUM = 4_000_000 * 1e18;  // 4M tokens

    enum ProposalState {
        Pending,
        Active,
        Canceled,
        Defeated,
        Succeeded,
        Queued,
        Expired,
        Executed
    }

    struct Proposal {
        uint256 id;
        address proposer;
        address[] targets;
        uint256[] values;
        bytes[] calldatas;
        string description;
        uint256 voteStart;
        uint256 voteEnd;
        uint256 forVotes;
        uint256 againstVotes;
        uint256 abstainVotes;
        bool executed;
    }

    mapping(uint256 => Proposal) public proposals;
    mapping(uint256 => mapping(address => bool)) public hasVoted;

    uint256 private _proposalCount;

    event ProposalCreated(
        uint256 indexed id,
        address indexed proposer,
        address[] targets,
        uint256[] values,
        bytes[] calldatas,
        string description
    );
    event VoteCast(address indexed voter, uint256 indexed proposalId, uint8 support, uint256 weight);
    event ProposalExecuted(uint256 indexed id);

    constructor(address token_) {
        token = IGovernanceToken(token_);
    }

    // Creer une proposition
    function propose(
        address[] memory targets,
        uint256[] memory values,
        bytes[] memory calldatas,
        string memory description
    ) public returns (uint256) {
        require(
            token.getPastVotes(msg.sender, block.number - 1) >= PROPOSAL_THRESHOLD,
            "Below proposal threshold"
        );

        uint256 proposalId = ++_proposalCount;

        proposals[proposalId] = Proposal({
            id: proposalId,
            proposer: msg.sender,
            targets: targets,
            values: values,
            calldatas: calldatas,
            description: description,
            voteStart: block.number + VOTING_DELAY,
            voteEnd: block.number + VOTING_DELAY + VOTING_PERIOD,
            forVotes: 0,
            againstVotes: 0,
            abstainVotes: 0,
            executed: false
        });

        emit ProposalCreated(proposalId, msg.sender, targets, values, calldatas, description);
        return proposalId;
    }

    // Voter
    function castVote(uint256 proposalId, uint8 support) public {
        require(state(proposalId) == ProposalState.Active, "Voting not active");
        require(!hasVoted[proposalId][msg.sender], "Already voted");

        Proposal storage proposal = proposals[proposalId];
        uint256 weight = token.getPastVotes(msg.sender, proposal.voteStart);

        hasVoted[proposalId][msg.sender] = true;

        if (support == 0) {
            proposal.againstVotes += weight;
        } else if (support == 1) {
            proposal.forVotes += weight;
        } else {
            proposal.abstainVotes += weight;
        }

        emit VoteCast(msg.sender, proposalId, support, weight);
    }

    // Etat de la proposition
    function state(uint256 proposalId) public view returns (ProposalState) {
        Proposal storage proposal = proposals[proposalId];

        if (proposal.executed) return ProposalState.Executed;
        if (block.number < proposal.voteStart) return ProposalState.Pending;
        if (block.number < proposal.voteEnd) return ProposalState.Active;

        uint256 totalVotes = proposal.forVotes + proposal.againstVotes + proposal.abstainVotes;
        if (totalVotes < QUORUM) return ProposalState.Defeated;
        if (proposal.againstVotes >= proposal.forVotes) return ProposalState.Defeated;

        return ProposalState.Succeeded;
    }

    // Executer la proposition
    function execute(uint256 proposalId) public {
        require(state(proposalId) == ProposalState.Succeeded, "Proposal not succeeded");

        Proposal storage proposal = proposals[proposalId];
        proposal.executed = true;

        for (uint256 i = 0; i < proposal.targets.length; i++) {
            (bool success, ) = proposal.targets[i].call{value: proposal.values[i]}(
                proposal.calldatas[i]
            );
            require(success, "Execution failed");
        }

        emit ProposalExecuted(proposalId);
    }
}
'''

print("Governor simplifie:")
print(GOVERNOR_CONTRACT)

Governor simplifie:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

interface IGovernanceToken {
    function getVotes(address account) external view returns (uint256);
    function getPastVotes(address account, uint256 blockNumber) external view returns (uint256);
}

contract SimpleGovernor {
    IGovernanceToken public immutable token;

    uint256 public constant VOTING_DELAY = 1;        // Blocs avant vote
    uint256 public constant VOTING_PERIOD = 50400;   // ~1 semaine (12s blocks)
    uint256 public constant PROPOSAL_THRESHOLD = 100 * 1e18;  // Min tokens pour proposer
    uint256 public constant QUORUM = 4_000_000 * 1e18;  // 4M tokens

    enum ProposalState {
        Pending,
        Active,
        Canceled,
        Defeated,
        Succeeded,
        Queued,
        Expired,
        Executed
    }

    struct Proposal {
        uint256 id;
        address proposer;
        address[] targets;
        uint256[] values;
        bytes[] calldatas;
        string de

---

## 3. Timelock Controller

Le timelock ajoute un delai avant l'execution pour la securite.

In [3]:
# Timelock simplifie
TIMELOCK_CONTRACT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

contract SimpleTimelock {
    uint256 public constant MIN_DELAY = 1 days;
    uint256 public constant MAX_DELAY = 30 days;
    uint256 public constant GRACE_PERIOD = 14 days;

    address public admin;
    uint256 public delay;

    mapping(bytes32 => bool) public queuedTransactions;

    event QueueTransaction(
        bytes32 indexed txHash,
        address indexed target,
        uint256 value,
        bytes data,
        uint256 eta
    );
    event ExecuteTransaction(
        bytes32 indexed txHash,
        address indexed target,
        uint256 value,
        bytes data
    );
    event CancelTransaction(bytes32 indexed txHash);

    constructor(uint256 delay_) {
        require(delay_ >= MIN_DELAY, "Delay too short");
        require(delay_ <= MAX_DELAY, "Delay too long");
        admin = msg.sender;
        delay = delay_;
    }

    // Hash unique pour chaque transaction
    function getTxHash(
        address target,
        uint256 value,
        bytes calldata data,
        uint256 eta
    ) public pure returns (bytes32) {
        return keccak256(abi.encode(target, value, data, eta));
    }

    // Mettre en file d'attente
    function queue(
        address target,
        uint256 value,
        bytes calldata data,
        uint256 eta
    ) public returns (bytes32) {
        require(msg.sender == admin, "Not admin");
        require(eta >= block.timestamp + delay, "ETA too soon");

        bytes32 txHash = getTxHash(target, value, data, eta);
        require(!queuedTransactions[txHash], "Already queued");

        queuedTransactions[txHash] = true;
        emit QueueTransaction(txHash, target, value, data, eta);

        return txHash;
    }

    // Executer apres le delai
    function execute(
        address target,
        uint256 value,
        bytes calldata data,
        uint256 eta
    ) public payable {
        require(msg.sender == admin, "Not admin");

        bytes32 txHash = getTxHash(target, value, data, eta);
        require(queuedTransactions[txHash], "Not queued");
        require(block.timestamp >= eta, "Too early");
        require(block.timestamp <= eta + GRACE_PERIOD, "Expired");

        queuedTransactions[txHash] = false;

        (bool success, ) = target.call{value: value}(data);
        require(success, "Execution failed");

        emit ExecuteTransaction(txHash, target, value, data);
    }

    // Annuler
    function cancel(bytes32 txHash) public {
        require(msg.sender == admin, "Not admin");
        require(queuedTransactions[txHash], "Not queued");

        queuedTransactions[txHash] = false;
        emit CancelTransaction(txHash);
    }
}
'''

print("Timelock Controller:")
print(TIMELOCK_CONTRACT)

Timelock Controller:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

contract SimpleTimelock {
    uint256 public constant MIN_DELAY = 1 days;
    uint256 public constant MAX_DELAY = 30 days;
    uint256 public constant GRACE_PERIOD = 14 days;

    address public admin;
    uint256 public delay;

    mapping(bytes32 => bool) public queuedTransactions;

    event QueueTransaction(
        bytes32 indexed txHash,
        address indexed target,
        uint256 value,
        bytes data,
        uint256 eta
    );
    event ExecuteTransaction(
        bytes32 indexed txHash,
        address indexed target,
        uint256 value,
        bytes data
    );
    event CancelTransaction(bytes32 indexed txHash);

    constructor(uint256 delay_) {
        require(delay_ >= MIN_DELAY, "Delay too short");
        require(delay_ <= MAX_DELAY, "Delay too long");
        admin = msg.sender;
        delay = delay_;
    }

    // Hash unique pour chaque transaction
    function getTxHash(
  

---

## 4. Exercices

### Exercice 1 : Calcul du quorum

Ecrivez une fonction pour verifier si le quorum est atteint.

In [ ]:
# Exercice 1 : Calcul du quorum
def check_quorum(for_votes, against_votes, abstain_votes, total_supply, quorum_percent=4):
    """
    Verifie si le quorum est atteint
    
    Quorum = (for + against + abstain) >= quorum_percent * total_supply / 100
    
    Args:
        for_votes: Nombre de votes pour
        against_votes: Nombre de votes contre
        abstain_votes: Nombre d abstentions
        total_supply: Supply totale de tokens
        quorum_percent: Pourcentage requis pour le quorum
    
    Returns:
        dict avec:
        - quorum_reached (bool)
        - total_votes
        - quorum_threshold
        - participation_percent
        - for_percent
        - against_percent
    """
    # TODO: Calculer le total des votes
    # TODO: Calculer le seuil du quorum (total_supply * quorum_percent / 100)
    # TODO: Determiner si le quorum est atteint
    # TODO: Calculer le pourcentage de participation
    # TODO: Calculer les pourcentages pour/contre
    # TODO: Retourner le dictionnaire avec tous les resultats
    pass

# Test (decommenter apres implementation)
# total_supply = 100_000_000 * 1e18  # 100M tokens
# result = check_quorum(
#     for_votes=3_500_000 * 1e18,
#     against_votes=1_000_000 * 1e18,
#     abstain_votes=500_000 * 1e18,
#     total_supply=total_supply
# )
# 
# print(f"Quorum atteint: {result["quorum_reached"]}")
# print(f"Participation: {result["participation_percent"]:.2f}%")
# print(f"Pour: {result["for_percent"]:.1f}%")
# print(f"Contre: {result["against_percent"]:.1f}%")

### Exercice 2 : Simulation de vote

Simulez un processus de gouvernance complet.

In [ ]:
# Exercice 2 : Simulation de vote
class ProposalSimulation:
    """Simule un processus de gouvernance complet."""
    
    def __init__(self, quorum=4_000_000, threshold=0.5):
        """
        Args:
            quorum: Nombre minimum de votes requis
            threshold: Seuil de majorite (0.5 = 50%)
        """
        self.quorum = quorum
        self.threshold = threshold
        # TODO: Initialiser les compteurs de votes (for, against, abstain)
        # TODO: Initialiser un dictionnaire pour tracker les votants

    def vote(self, voter, weight, support):
        """
        Enregistre un vote.
        
        Args:
            voter: Identifiant du votant
            weight: Poids du vote (nombre de tokens)
            support: 0 = contre, 1 = pour, 2 = abstention
        """
        # TODO: Verifier que le votant n a pas deja vote
        # TODO: Enregistrer le vote dans le dictionnaire
        # TODO: Ajouter le poids au compteur correspondant (for/against/abstain)
        pass

    def get_result(self):
        """
        Determine le resultat du vote.
        
        Returns:
            str: "SUCCEEDED", "DEFEATED - Quorum non atteint", 
                 ou "DEFEATED - Majorite contre"
        """
        # TODO: Calculer le total des votes
        # TODO: Verifier si le quorum est atteint
        # TODO: Verifier si la majorite est pour
        # TODO: Retourner le resultat
        pass

# Test (decommenter apres implementation)
# sim = ProposalSimulation()
# sim.vote("Alice", 1_000_000, 1)    # Pour
# sim.vote("Bob", 2_000_000, 1)      # Pour
# sim.vote("Charlie", 1_500_000, 0)  # Contre
# sim.vote("Dave", 500_000, 2)       # Abstain
# sim.vote("Eve", 800_000, 1)        # Pour
# 
# print(f"Pour: {sim.for_votes:,}")
# print(f"Contre: {sim.against_votes:,}")
# print(f"Abstention: {sim.abstain_votes:,}")
# print(f"Resultat: {sim.get_result()}")

---

## 5. Resume

| Composant | Role |
|-----------|------|
| GovernanceToken | Token avec voting power |
| Governor | Gestion des propositions et votes |
| Timelock | Delai de securite avant execution |

### Parametres cles
- **Voting Delay** : Blocs avant debut du vote
- **Voting Period** : Duree du vote
- **Quorum** : Participation minimum requise
- **Proposal Threshold** : Tokens min pour proposer

---

**Notebook suivant** : [SC-8-Account-Abstraction](SC-8-Account-Abstraction.ipynb)